In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("🧪 1. جاري إعداد بيانات النطاقات الكيميائية والمغذيات لـ Model 2...")

# إعداد ملفات المحاصيل لـ Model 2
crop_profiles_m2 = {
    'Wheat (قمح)': {'N': (80, 120), 'P': (40, 60), 'K': (30, 50), 'pH': (6.0, 7.0)},
    'Barley (شعير)': {'N': (60, 90), 'P': (30, 50), 'K': (25, 45), 'pH': (6.2, 7.8)},
    'Soybean (فول صويا)': {'N': (20, 40), 'P': (60, 90), 'K': (50, 80), 'pH': (6.0, 7.5)},
    'Corn (ذرة صفراء)': {'N': (100, 150), 'P': (45, 70), 'K': (40, 65), 'pH': (5.8, 7.0)},
    'Sunflower (عباد الشمس)': {'N': (50, 80), 'P': (40, 60), 'K': (60, 90), 'pH': (6.5, 7.5)},
    'Tomato (طماطم)': {'N': (70, 110), 'P': (40, 70), 'K': (40, 60), 'pH': (6.0, 6.8)},
    'Potato (بطاطس)': {'N': (40, 70), 'P': (40, 60), 'K': (45, 60), 'pH': (5.0, 6.5)}
}

# توليد مجموعة البيانات
np.random.seed(100)
data_m2 = []
for crop, bounds in crop_profiles_m2.items():
    for _ in range(300):
        n = np.random.uniform(*bounds['N']) + np.random.normal(0, 2)
        p = np.random.uniform(*bounds['P']) + np.random.normal(0, 2)
        k = np.random.uniform(*bounds['K']) + np.random.normal(0, 2)
        ph = np.random.uniform(*bounds['pH']) + np.random.normal(0, 0.1)
        data_m2.append([n, p, k, ph, crop])

df_m2 = pd.DataFrame(data_m2, columns=['N', 'P', 'K', 'pH', 'Crop'])

X_m2 = df_m2[['N', 'P', 'K', 'pH']]
y_m2 = df_m2['Crop']

le_m2 = LabelEncoder()
y_encoded_m2 = le_m2.fit_transform(y_m2)

X_train_m2, X_test_m2, y_train_m2, y_test_m2 = train_test_split(X_m2, y_encoded_m2, test_size=0.2, random_state=42)

# 🤖 2. تدريب وتقييم الخوارزميات
print("\n🤖 2. بدء تدريب وتقييم النماذج لـ Model 2...")

rf_model_m2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_m2.fit(X_train_m2, y_train_m2)
rf_acc_m2 = accuracy_score(y_test_m2, rf_model_m2.predict(X_test_m2)) * 100

xgb_model_m2 = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model_m2.fit(X_train_m2, y_train_m2)
xgb_acc_m2 = accuracy_score(y_test_m2, xgb_model_m2.predict(X_test_m2)) * 100

print(f"🌲 دقة Random Forest: {rf_acc_m2:.2f}%")
print(f"🚀 دقة XGBoost: {xgb_acc_m2:.2f}%")

best_model_m2 = xgb_model_m2 if xgb_acc_m2 >= rf_acc_m2 else rf_model_m2
best_name_m2 = "XGBoost" if xgb_acc_m2 >= rf_acc_m2 else "Random Forest"
print(f"✅ تم اعتماد النموذج الأفضل لـ Model 2: {best_name_m2}")

🧪 1. جاري إعداد بيانات النطاقات الكيميائية والمغذيات لـ Model 2...

🤖 2. بدء تدريب وتقييم النماذج لـ Model 2...
🌲 دقة Random Forest: 88.81%
🚀 دقة XGBoost: 88.10%
✅ تم اعتماد النموذج الأفضل لـ Model 2: Random Forest


In [2]:
!pip install fastapi uvicorn nest_asyncio pydantic

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import threading
import os

# إغلاق أي سيرفر قديم يعمل على المنفذ 8000
os.system("pkill -f uvicorn")

app = FastAPI(
    title="HAK Model 2 - Chemical Nutrient Analysis API",
    description="API لتنبؤ المحاصيل والتوصيات بناءً على الخواص الكيميائية للتربة"
)

class NutrientInput(BaseModel):
    n: float
    p: float
    k: float
    ph: float

@app.post("/predict-model2")
async def predict_model2_endpoint(input_data: NutrientInput):
    try:
        sample = pd.DataFrame([[input_data.n, input_data.p, input_data.k, input_data.ph]],
                              columns=['N', 'P', 'K', 'pH'])

        pred_idx = best_model_m2.predict(sample)[0]
        probs = best_model_m2.predict_proba(sample)[0]

        recommended_crop = le_m2.inverse_transform([pred_idx])[0]
        confidence = float(probs[pred_idx] * 100)

        sorted_indices = np.argsort(probs)
        unsuitable_indices = sorted_indices[:2]
        unsuitable_crops = le_m2.inverse_transform(unsuitable_indices).tolist()

        return {
            "status": "success",
            "model_used": best_name_m2,
            "inputs": {
                "N": input_data.n,
                "P": input_data.p,
                "K": input_data.k,
                "pH": input_data.ph
            },
            "result": {
                "recommended_crop": str(recommended_crop),
                "confidence_percentage": round(confidence, 2),
                "unsuitable_crops": unsuitable_crops
            }
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = threading.Thread(target=start_server)
server_thread.daemon = True
server_thread.start()

print("⚡ تم تشغيل سيرفر Model 2 بنجاح في الخلفية!")

⚡ تم تشغيل سيرفر Model 2 بنجاح في الخلفية!


In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("🌐 جاري إنشاء الرابط المستقر للنموذج الثاني...")
print("⬇️ ابحثي في المخرجات أدناه عن رابط ينتهي بـ trycloudflare.com ⬇️")

!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000

🌐 جاري إنشاء الرابط المستقر للنموذج الثاني...
⬇️ ابحثي في المخرجات أدناه عن رابط ينتهي بـ trycloudflare.com ⬇️
2026-08-28T16:31:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-28T16:31:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-28T16:31:54Z INF +--------------------------------------------------------------------------------------------+
2026-08-28T16:31:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take s